# Silver 1 — process_invoice
Reads raw files from Bronze, cleans and normalises, MERGEs into `Tables/silver/staging_<source>`.

**Parameters injected by Airflow:** `source_config` (JSON string), `env`

In [ ]:
# Fabric Notebook — Silver 1: Clean invoice file and MERGE into staging table
# ===========================================================================
# Triggered by: medallion_pipeline DAG → FabricRunItemOperator
# Parameters:
#   source_config : JSON string of one row from sources.json
#   env           : "dev" | "prod"
#
# Responsibility:
#   1. Read the raw file(s) from Bronze Lakehouse (CSV/Parquet via Spark, Excel via pandas)
#   2. Parse, validate, clean and normalise column names
#   3. MERGE into Silver Lakehouse Delta table: Tables/silver/staging_{source_name}
#   4. Log result to data_ops.pipeline_run_log

import io
import json
import sys
from functools import reduce
from pathlib import Path

from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F

### Parameters

In [ ]:
try:
    source_config_str = source_config  # noqa: F821
    env = env  # noqa: F821
except NameError:
    source_config_str = "{}"
    env = "dev"

src: dict = json.loads(source_config_str)
source_name: str = src["source_name"]
bronze_path: str = src["bronze_path"]
silver1_table: str = src["silver1_table"]
file_format: str = src["file_format"]  # csv | excel | parquet | json | avro
extra_params: dict = src.get("extra_params", {})

FORMAT_GLOB = {
    "csv":     "*.csv",
    "excel":   "*.xlsx",
    "parquet": "*.parquet",
    "json":    "*.json",
    "avro":    "*.avro",
}
file_glob = FORMAT_GLOB.get(file_format, f"*.{file_format}")

### Shared lib

In [ ]:
sys.path.insert(0, "/lakehouse/default/Files/shared")
import shared_functions as gf

### Config

In [ ]:
config_path = Path(__file__).parent.parent.parent.parent / "config" / f"{env}.json"
cfg: dict = json.loads(config_path.read_text())

WORKSPACE   = cfg["workspace_name"]
LAKEHOUSE   = cfg["lakehouse"]
DW_ENDPOINT = cfg["gold_warehouse_sql_endpoint"]
DW_DATABASE = cfg["gold_warehouse"]

try:
    run_id = mssparkutils.notebook.run("_get_run_id")  # noqa: F821
except Exception:
    run_id = "local"

print(f"[silver1/process_invoice] source={source_name} table={silver1_table}")

spark = SparkSession.builder.getOrCreate()

### 1. List unprocessed bronze files for this source

In [ ]:
with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
    records = gf.query_to_records(
        conn,
        f"SELECT file_name FROM data_ops.pipeline_run_log "
        f"WHERE source_name = '{source_name}' AND layer = 'silver1' AND status = 'success'"
    )
    processed_files = {r["file_name"] for r in records}

bronze_files = gf.list_files(
    workspace_name=WORKSPACE,
    lakehouse_name=LAKEHOUSE,
    prefix=bronze_path,
    pattern=file_glob,
)

new_files = [f for f in bronze_files if f.split("/")[-1] not in processed_files]
print(f"  Bronze files: {len(bronze_files)} total, {len(new_files)} unprocessed")

if not new_files:
    print("  Nothing to process")
    mssparkutils.notebook.exit(json.dumps({"status": "no_new_files", "rows": 0}))  # noqa: F821

### 2. Read each file into a Spark DataFrame

In [ ]:
all_frames = []

for abfss_path in new_files:
    file_name = abfss_path.split("/")[-1]
    print(f"  Reading {file_name} ({file_format}) ...")

    if file_format == "csv":
        sdf = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(abfss_path)
        )
    elif file_format == "parquet":
        sdf = spark.read.parquet(abfss_path)
    elif file_format == "excel":
        import pandas as pd
        raw_bytes = gf.read_file_bytes(abfss_path)
        pdf = pd.read_excel(io.BytesIO(raw_bytes), sheet_name=extra_params.get("sheet", 0))
        sdf = spark.createDataFrame(pdf)
    elif file_format == "json":
        sdf = spark.read.option("multiline", True).json(abfss_path)
    elif file_format == "avro":
        sdf = spark.read.format("avro").load(abfss_path)
    else:
        raise ValueError(f"Unsupported file_format: '{file_format}'")

    sdf = (
        sdf
        .withColumn("_source_name", F.lit(source_name))
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_run_id",      F.lit(run_id))
    )
    all_frames.append(sdf)

if not all_frames:
    raise RuntimeError("No frames loaded — check file parsing")

combined = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), all_frames)
print(f"  Raw rows: {combined.count()}, columns: {len(combined.columns)}")

### 3. Clean

In [ ]:
# 3a: normalise column names, trim strings, cast booleans, truncate strings
combined = gf.spark_clean_dataframe(combined)

# 3b: extract numeric values from currency/percent string columns
amount_cols = extra_params.get("amount_cols", [])
if amount_cols:
    combined = gf.spark_extract_numeric(combined, amount_cols)
    print(f"  Extracted numeric: {amount_cols}")

# 3c: parse date columns to Date type
date_cols   = extra_params.get("date_cols", [])
date_format = extra_params.get("date_format", None)
if date_cols:
    combined = gf.spark_parse_dates(combined, date_cols, fmt=date_format)
    print(f"  Parsed dates: {date_cols} (fmt={date_format or 'auto'})")

# 3d: coerce incoming columns to the existing Delta table schema (if table exists)
silver_abfss = gf.onelake_abfss(WORKSPACE, LAKEHOUSE, f"Tables/silver/{silver1_table}")
try:
    existing_schema = spark.read.format("delta").load(silver_abfss).schema
    table_exists = True
    combined = gf.spark_cast_to_schema(combined, existing_schema)
    print(f"  Cast to existing Delta schema ({len(existing_schema.fields)} columns)")
except Exception:
    table_exists = False

# 3e: drop rows where every non-metadata column is null
data_cols = [c for c in combined.columns if not c.startswith("_")]
combined = combined.filter(
    sum(F.col(c).isNotNull().cast("int") for c in data_cols) > 0
)

rows_written = combined.count()
print(f"  After cleaning: {rows_written} rows")

### 4. Write to Silver Lakehouse as Delta table

In [ ]:
if not table_exists:
    print(f"  Creating new Delta table at {silver_abfss}")
    combined.write.format("delta").mode("overwrite").save(silver_abfss)
else:
    combined.createOrReplaceTempView("incoming")

    spark.sql(f"""
        MERGE INTO delta.`{silver_abfss}` AS target
        USING incoming AS source
        ON target._source_file = source._source_file
           AND target._source_name = source._source_name
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"  MERGE complete into {silver_abfss}")

### 5. Log

In [ ]:
with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
    for file_name in [f.split("/")[-1] for f in new_files]:
        gf.log_run(
            conn,
            source_name=source_name,
            layer="silver1",
            status="success",
            rows_processed=rows_written,
            run_id=run_id,
        )

result = {"status": "success", "source_name": source_name, "rows": rows_written, "table": silver1_table}
print(f"  Result: {result}")
mssparkutils.notebook.exit(json.dumps(result))  # noqa: F821